# ImmigrationNavigator — Data Collection

**UC Berkeley MIDS Capstone 2026** — Team: Ale, Clover, Duc, Rohan

---

## What this notebook does

Collects and saves the full USCIS immigration corpus to S3. This is the foundation of the RAG pipeline — everything downstream depends on this data.

## Data sources

| # | Source | Method | Why we use it |
|---|--------|--------|---------------|
| 1 | USCIS Policy Manual | Clover's section parser with HTTP caching | Main policy text: F-1, OPT, STEM OPT, H-1B |
| 2 | USCIS AFM PDFs | Direct PDF download | H-1B chapters not yet migrated to Policy Manual |
| 3 | State Dept Visa Bulletin | HTML scraper | Monthly priority dates for green card timeline |
| 4 | Federal Register | REST API | Recent policy change notices |

## Output

`s3://immigration-navigator-data/raw_docs.json` — list of document dicts with keys: `source`, `label`, `url`, `text`, `word_count`, `char_count`, `scraped_at`

## Run order

Run all cells top to bottom. Each cell depends on the one above it.

## 1. Install Dependencies

In [1]:
!pip install requests beautifulsoup4 pdfplumber pandas tiktoken boto3 -q

## 2. Imports and Configuration

Sets up all imports and constants used throughout the notebook.

- **AWS boto3** — reads from S3 and Secrets Manager (no credentials in code)
- **USCIS_URL** — single HTML export endpoint that contains the full Policy Manual
- **CACHE_FILE** — stores ETag/Last-Modified headers so we only re-download when content changes
- **headers** — browser User-Agent to avoid bot-blocking on government sites

In [2]:
import os, re, json, time, hashlib, requests, urllib.request, boto3
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
import tiktoken

# AWS setup
def get_secret(secret_name):
    client = boto3.client("secretsmanager", region_name="us-east-1")
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response["SecretString"])

s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"

# USCIS full policy manual — single HTML export endpoint
USCIS_URL = "https://www.uscis.gov/book/export/html/68600"

# Cache file — ETag/Last-Modified so we only re-fetch when content changes
CACHE_FILE = "/tmp/uscis_cache.json"

# Index storage and chunking config
PERSIST_DIR = "/home/sagemaker-user/index_store"
CHUNK_SIZE_TOKENS = 512
CHUNK_OVERLAP_TOKENS = 64

# General browser headers — used by Visa Bulletin and other scrapers
headers = {"User-Agent": "Mozilla/5.0"}

print("Setup complete")
print(f"  USCIS_URL   : {USCIS_URL}")
print(f"  S3_BUCKET   : {S3_BUCKET}")
print(f"  CACHE_FILE  : {CACHE_FILE}")
print(f"  Chunk size  : {CHUNK_SIZE_TOKENS} tokens, overlap {CHUNK_OVERLAP_TOKENS}")


Setup complete
  USCIS_URL   : https://www.uscis.gov/book/export/html/68600
  S3_BUCKET   : immigration-navigator-data
  CACHE_FILE  : /tmp/uscis_cache.json
  Chunk size  : 512 tokens, overlap 64


## 3. Fetch USCIS Policy Manual

Downloads the full USCIS Policy Manual as a single HTML file.

**How the caching works:** On the first run, it downloads the full HTML (~5MB) and saves an ETag fingerprint. On subsequent runs, it sends that ETag to USCIS — if the content hasn't changed, USCIS returns a `304 Not Modified` response and we skip the download entirely. This is Clover's contribution and is significantly better than re-downloading every time.

> Set `force_refresh=True` to bypass the cache and always re-fetch.

In [3]:
# ── Cache helpers ──

def load_cache() -> dict:
    """Load cached metadata (ETag, Last-Modified, file path) from disk."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    return {}


def save_cache(metadata: dict) -> None:
    """Persist cache metadata to disk."""
    with open(CACHE_FILE, "w") as f:
        json.dump(metadata, f, indent=2)

In [4]:
def fetch_uscis_manual(force_refresh: bool = False) -> str:
    """
    Fetch the USCIS Policy Manual HTML.

    Uses HTTP ETag / Last-Modified headers so we only re-download
    when the document has actually changed.

    Args:
        force_refresh: If True, ignores cache and always re-fetches.

    Returns:
        Raw HTML string of the full policy manual.
    """
    cache = load_cache()
    headers = {"User-Agent": "Mozilla/5.0 (RAG research tool)"}

    # Add conditional request headers if we have cached validators
    if not force_refresh:
        if "etag" in cache:
            headers["If-None-Match"] = cache["etag"]
        elif "last_modified" in cache:
            headers["If-Modified-Since"] = cache["last_modified"]

    print(f"Fetching from {USCIS_URL} ...")
    response = requests.get(USCIS_URL, headers=headers, timeout=60)

    # 304 Not Modified — document hasn't changed, use cached HTML
    if response.status_code == 304:
        print("Document unchanged (304 Not Modified). Using cached version.")
        with open(cache["html_file"], "r", encoding="utf-8") as f:
            return f.read()

    response.raise_for_status()
    html = response.text
    print(f"Fetched {len(html):,} characters of HTML.")

    # Save raw HTML and update cache metadata
    html_file = "uscis_raw.html"
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)

    cache.update({
        "etag": response.headers.get("ETag", ""),
        "last_modified": response.headers.get("Last-Modified", ""),
        "html_file": html_file,
        "fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "content_hash": hashlib.md5(html.encode()).hexdigest(),
    })
    save_cache(cache)
    print(f"Cached to '{html_file}'. Metadata saved to '{CACHE_FILE}'.")
    return html


# Run it — set force_refresh=True to bypass the cache and always re-fetch
html = fetch_uscis_manual(force_refresh=False)
print(f"\nHTML ready: {len(html):,} characters")

Fetching from https://www.uscis.gov/book/export/html/68600 ...
Fetched 11,691,266 characters of HTML.


Cached to 'uscis_raw.html'. Metadata saved to '/tmp/uscis_cache.json'.

HTML ready: 11,691,266 characters


## 4. Parse HTML into Structured Sections

Converts the raw HTML into structured `Section` objects by walking the heading hierarchy (H1–H4).

Each section gets:
- **title** — the heading text
- **text** — all body content under that heading
- **citation_url** — a direct deep-link to that section on uscis.gov
- **breadcrumb** — the ancestor heading path (e.g. Volume 2 > Part F > Chapter 5)

This structure-aware approach produces more semantically coherent chunks than flat text splitting, because we never split content that belongs to the same section.

In [5]:
@dataclass
class Section:
    """One parsed section of the USCIS policy manual."""
    title: str                           # Section heading text
    level: int                           # Heading level (1=H1, 2=H2, etc.)
    text: str                            # Body text under this heading
    citation_url: str                    # Deep-link URL for citing this section
    breadcrumb: list[str] = field(default_factory=list)  # Ancestor heading titles

In [6]:
def parse_sections(html: str) -> list[Section]:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup.find_all(["nav", "script", "style", "header", "footer"]):
        tag.decompose()

    sections: list[Section] = []
    heading_tags = ["h1", "h2", "h3", "h4"]
    breadcrumb_stack: list[tuple[int, str]] = []
    current_chapter_url = "https://www.uscis.gov/policy-manual"
    base = "https://www.uscis.gov"

    all_headings = soup.find_all(heading_tags)

    for i, heading in enumerate(all_headings):
        if len(heading.name) < 2:
            continue

        level = int(heading.name[1])
        title = heading.get_text(strip=True).replace('\xa0', ' ')
        inner_links = heading.find_all("a")
        heading_classes = heading.get("class") or []

        # ── Pattern 1: bookmark <h2> — update chapter URL and skip ──
        is_bookmark = False
        for a in inner_links:
            if "bookmark" in (a.get("rel") or []) and a.get("href"):
                current_chapter_url = base + a["href"]
                is_bookmark = True
                break
        if is_bookmark:
            continue

        # ── Pattern 2: level__title <h2> — TOC navigation, skip entirely ──
        # These are just the table of contents links, not real content
        if "level__title" in heading_classes:
            continue

        # ── Pattern 3: book-node-heading <h1> — actual chapter/part title ──
        # Peek at the next heading to find its bookmark URL since the
        # bookmark <h2> comes AFTER the title <h1> in the HTML
        if "book-node-heading" in heading_classes:
            for next_heading in all_headings[i+1:i+3]:
                next_links = next_heading.find_all("a")
                for a in next_links:
                    if "bookmark" in (a.get("rel") or []) and a.get("href"):
                        current_chapter_url = base + a["href"]
                        break
                break
            citation_url = current_chapter_url

        # ── Pattern 4: ck-anchor — section level heading ──
        else:
            citation_url = current_chapter_url
            for a in inner_links:
                if "ck-anchor" in (a.get("class") or []) and a.get("id"):
                    citation_url = f"{current_chapter_url}#{a['id']}"
                    break

        # ── Collect body text ──
        body_parts = []
        for sibling in heading.find_next_siblings():
            if sibling.name in heading_tags:
                break  # Stop at any same-or-higher heading
            # Collect direct content tags only — don't recurse into nested sections
            if sibling.name in ["p", "ul", "ol", "table", "div", "blockquote"]:
                # Check if this sibling contains a heading — if so, extract only
                # the text that comes before the first nested heading
                nested = sibling.find(heading_tags)
                if nested:
                    # Get text up to the nested heading only
                    for child in sibling.children:
                        if hasattr(child, 'name') and child.name in heading_tags:
                            break
                        text = child.get_text(separator=" ", strip=True) if hasattr(child, 'get_text') else str(child).strip()
                        if text:
                            body_parts.append(text)
                    break
                else:
                    text = sibling.get_text(separator=" ", strip=True)
                    if text:
                        body_parts.append(text)

        body_text = " ".join(body_parts).replace('\xa0', ' ')

        breadcrumb_stack = [(l, t) for l, t in breadcrumb_stack if l < level]
        current_breadcrumb = [t for _, t in breadcrumb_stack]
        breadcrumb_stack.append((level, title))

        sections.append(Section(
            title=title,
            level=level,
            text=body_text,
            citation_url=citation_url,
            breadcrumb=current_breadcrumb,
        ))

    return sections

sections = parse_sections(html)
print(f"Parsed {len(sections):,} sections from the USCIS manual.")

# Preview the first 5 sections
for s in sections[:5]:
    preview = s.text[:80].replace("\n", " ") + "..." if len(s.text) > 80 else s.text
    print(f"  H{s.level} | {s.title[:60]} | {preview}")

Parsed 4,341 sections from the USCIS manual.
  H1 | Policy Manual | 
  H4 | About the Policy Manual | The USCIS Policy Manual is the agency’s centralized online repository for USCIS’...
  H4 | Adjudicator's Field Manual Transition | USCIS is retiring its Adjudicator's Field Manual (AFM), a collection of our immi...
  H3 | Get Updates by Email | Email Address e.g. name@email.com
  H1 | Search | 


## 5. Convert Sections to Document Dicts

Transforms the parsed `Section` objects into the standard document dict format used by all downstream notebooks. Skips empty sections (headings with no body text).

In [7]:
# Convert parsed Section objects to document dicts for the RAG corpus
uscis_docs = []
for section in sections:
    if not section.text.strip():
        continue
    uscis_docs.append({
        "source":     "USCIS Policy Manual",
        "label":      section.title[:50] if section.title else "untitled",
        "url":        section.citation_url,
        "text":       section.text,
        "scraped_at": datetime.now().isoformat(),
        "char_count": len(section.text),
        "word_count": len(section.text.split()),
    })

print(f"uscis_docs: {len(uscis_docs)} sections")
print(f"Total words: {sum(d['word_count'] for d in uscis_docs):,}")


uscis_docs: 3415 sections
Total words: 915,302


## 6. H-1B Content via AFM PDFs

USCIS has not finished migrating H-1B content into the online Policy Manual. The detailed guidance still lives in the legacy **Adjudicator's Field Manual (AFM)**, published as public PDFs.

- **afm31** — H-1B petition requirements (~27K words)
- **afm34** — Other employment-authorized nonimmigrants (~4K words)

These are appended to `uscis_docs` so they flow through the same pipeline.

In [8]:
AFM_PDFS = {
    "h1b_afm_ch31": "https://www.uscis.gov/sites/default/files/document/policy-manual-afm/afm31-external.pdf",
    "h1b_afm_ch34": "https://www.uscis.gov/sites/default/files/document/policy-manual-afm/afm34-external.pdf",
}

h1b_afm_docs = []
for label, url in AFM_PDFS.items():
    try:
        # Download PDF to /tmp
        pdf_path = f"/tmp/{label}.pdf"
        urllib.request.urlretrieve(url, pdf_path)

        # Extract text from all pages
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join([page.extract_text() or "" for page in pdf.pages])

        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)

        h1b_afm_docs.append({
            "source":     "USCIS AFM",
            "label":      label,
            "url":        url,
            "text":       text,
            "scraped_at": datetime.now().isoformat(),
            "char_count": len(text),
            "word_count": len(text.split()),
        })
        print(f"OK {label}: {len(text.split())} words")

    except Exception as e:
        print(f"ERROR {label}: {e}")

uscis_docs += h1b_afm_docs
print(f"\nTotal USCIS + AFM docs : {len(uscis_docs)}")
print(f"Total words            : {sum(d['word_count'] for d in uscis_docs):,}")

OK h1b_afm_ch31: 27077 words


OK h1b_afm_ch34: 4439 words

Total USCIS + AFM docs : 3417
Total words            : 946,818


## 7. State Department Visa Bulletin

The Visa Bulletin is published monthly and contains priority date cutoffs for employment-based green cards. Relevant for users asking about the H-1B → green card timeline.

We scrape the 6 most recent bulletins dynamically — the index page is fetched first to discover the current URLs, so this never breaks when a new month is published.

In [9]:
def fetch_visa_bulletin(num_months=6):
    """
    Scrape the most recent Visa Bulletin issues from the State Department.

    Discovers bulletin URLs dynamically from the index page
    to avoid hardcoding URLs that change monthly.

    Args:
        num_months (int): Number of recent bulletins to scrape.

    Returns:
        list[dict]: List of document dicts, one per bulletin.
    """
    base_url  = "https://travel.state.gov"
    index_url = f"{base_url}/content/travel/en/legal/visa-law0/visa-bulletin.html"

    r = requests.get(index_url, headers=headers)
    soup = BeautifulSoup(r.content, "html.parser")

    # Discover bulletin URLs from the index page
    bulletin_urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "visa-bulletin-for" in href:
            full_url = base_url + href if href.startswith("/") else href
            if full_url not in bulletin_urls:
                bulletin_urls.append(full_url)

    print(f"Bulletins found: {len(bulletin_urls)} — scraping latest {num_months}")

    docs = []
    for url in bulletin_urls[:num_months]:
        try:
            r = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.content, "html.parser")

            main = (
                soup.find("div", class_="tsg-rwd-main-copy-body") or
                soup.find("main") or
                soup.find("article")
            )

            text = main.get_text(separator="\n", strip=True) if main else soup.get_text()
            text = re.sub(r'\n{3,}', '\n\n', text)
            text = re.sub(r'[ \t]+', ' ', text)

            label = url.split("visa-bulletin-for-")[-1].replace(".html", "")
            docs.append({
                "source":     "State Dept Visa Bulletin",
                "label":      f"visa_bulletin_{label}",
                "url":        url,
                "text":       text,
                "scraped_at": datetime.now().isoformat(),
                "char_count": len(text),
                "word_count": len(text.split()),
            })
            print(f"OK {label}: {len(text.split())} words")
            time.sleep(1)

        except Exception as e:
            print(f"ERROR {url}: {e}")

    return docs

visa_docs = fetch_visa_bulletin(num_months=6)
print(f"\nTotal Visa Bulletin docs: {len(visa_docs)}")

Bulletins found: 290 — scraping latest 6
OK june-2026: 3480 words


OK may-2026: 3195 words


OK april-2026: 3125 words


OK march-2026: 3184 words


OK february-2026: 3159 words


OK january-2026: 3179 words



Total Visa Bulletin docs: 6


## 8. Federal Register

Queries the Federal Register's free REST API for recent immigration-related rule notices.

These documents are short (~100–200 words each — title + abstract only) but useful for surfacing recent policy changes that may not yet be in the USCIS Policy Manual. Full text enrichment is a future improvement.

Keywords cover the full F-1 → OPT → H-1B pipeline.

In [10]:
def fetch_federal_register(keywords, max_results=20):
    """
    Query the Federal Register API for immigration-related notices.

    Returns title + abstract for each result. Full text is not
    fetched here to keep Week 1 scope manageable.

    Args:
        keywords    (list[str]): Search terms to query.
        max_results (int)      : Max documents per keyword.

    Returns:
        list[dict]: List of document dicts.
    """
    all_results = []

    for keyword in keywords:
        params = {
            "conditions[term]": keyword,
            "per_page":         max_results,
            "order":            "relevance",
            "fields[]": ["title", "abstract", "publication_date",
                         "html_url", "document_number", "type"],
        }
        r = requests.get(
            "https://www.federalregister.gov/api/v1/documents",
            params=params,
            timeout=10
        )
        results = r.json().get("results", [])
        print(f"OK '{keyword}': {len(results)} results")

        for item in results:
            text = f"{item.get('title','')}\n\n{item.get('abstract','') or ''}".strip()
            all_results.append({
                "source":           "Federal Register",
                "label":            "federal_register",
                "url":              item.get("html_url", ""),
                "text":             text,
                "publication_date": item.get("publication_date", ""),
                "scraped_at":       datetime.now().isoformat(),
                "word_count":       len(text.split()),
                "char_count":       len(text),
            })
        time.sleep(0.5)

    print(f"\nTotal Federal Register docs: {len(all_results)}")
    return all_results

# Keywords covering the full F-1 → OPT → H-1B pipeline
keywords = ["OPT practical training", "STEM OPT", "H-1B", "F-1 student"]
fed_docs = fetch_federal_register(keywords)

OK 'OPT practical training': 20 results


OK 'STEM OPT': 20 results


OK 'H-1B': 20 results


OK 'F-1 student': 20 results



Total Federal Register docs: 80


## 9. Combine, Save to S3, and EDA

Merges all four sources into a single list, normalizes field names across sources, saves to S3, and prints a summary of the corpus.

**The EDA tells us:**
- How many documents and words per source
- Which documents are largest (top 10)
- How many documents are ready for RAG chunking (>200 words)

> After this cell, `raw_docs.json` is in S3 and Notebook 2 (EDA) or Notebook 3 (RAG) can be run.

In [11]:
all_docs = uscis_docs + visa_docs + fed_docs

# Normalize fields
for d in all_docs:
    if "text" not in d or not d["text"]:
        d["text"] = f"{d.get('title','')}\n\n{d.get('abstract','')}".strip()
    if "word_count" not in d:
        d["word_count"] = len(d["text"].split())
    if "char_count" not in d:
        d["char_count"] = len(d["text"])

# Save to S3
s3.put_object(
    Bucket=S3_BUCKET,
    Key="raw_docs.json",
    Body=json.dumps(all_docs, indent=2).encode("utf-8")
)
print(f"Saved {len(all_docs)} documents to S3\n")

# EDA summary
df = pd.DataFrame([{
    "source":     d["source"],
    "label":      d["label"],
    "word_count": d["word_count"],
    "char_count": d["char_count"],
} for d in all_docs])

print("── Documents and words by source ──")
print(df.groupby("source").agg(
    documents=("label",       "count"),
    total_words=("word_count", "sum"),
    avg_words=("word_count",   "mean"),
).round(0))

print("\n── Top 10 documents by size ──")
print(df[["label", "word_count"]]
      .sort_values("word_count", ascending=False)
      .head(10)
      .to_string(index=False))

print("\n── Overall totals ──")
print(f"Total documents  : {len(all_docs)}")
print(f"Total words      : {df['word_count'].sum():,}")
print(f"Total characters : {df['char_count'].sum():,}")

rag_ready = df[df["word_count"] > 200]
print(f"\n── Ready for RAG chunking (>200 words) ──")
print(f"Documents : {len(rag_ready)}")
print(f"Words     : {rag_ready['word_count'].sum():,}")


Saved 3503 documents to S3

── Documents and words by source ──
                          documents  total_words  avg_words
source                                                     
Federal Register                 80         8820      110.0
State Dept Visa Bulletin          6        19322     3220.0
USCIS AFM                         2        31516    15758.0
USCIS Policy Manual            3415       915302      268.0

── Top 10 documents by size ──
                                       label  word_count
                                h1b_afm_ch31       27077
                                   Footnotes        5794
                                   Footnotes        5545
                                   Footnotes        4943
                                   Footnotes        4464
1. Initial Evidence of Extraordinary Ability        4463
                                h1b_afm_ch34        4439
                           Table of Contents        4390
                               